# Milvus Vector Database Connection & Operations Guide

This notebook demonstrates how to connect to a Milvus vector database and perform various operations including collection management, data insertion, indexing, and vector similarity search.

**Prerequisites:**
- Milvus server running at `milvus-api.local:19530`
- pymilvus library installed: `pip install pymilvus`

## 1. Import Required Libraries

We import the necessary Milvus client libraries for connection management, collection operations, and data manipulation.

In [39]:
from pymilvus import connections, Collection, MilvusException, FieldSchema, CollectionSchema, DataType, utility
import numpy as np
import random

## 2. Connect to Milvus Server

Establish a connection to the Milvus server. The `alias` parameter allows you to manage multiple connections.

In [40]:
def connect_to_milvus(alias='default', host='milvus-api.local', port=19530):
    """Connect to Milvus server"""
    try:
        connections.connect(alias=alias, host=host, port=port)
        print(f'Successfully connected to Milvus at {host}:{port}')
        return True
    except Exception as e:
        print(f'Failed to connect: {e}')
        return False

connect_to_milvus()

Successfully connected to Milvus at milvus-api.local:19530


True

## 3. Check Server Version and Status

Retrieve server information to verify connectivity and check the Milvus version.

In [41]:
def check_server_info():
    """Check Milvus server version and status"""
    try:
        server_info = utility.get_server_version()
        print(f'Milvus server version: {server_info}')
    except Exception as e:
        print(f'Could not retrieve server version: {e}')

check_server_info()

Milvus server version: pkg/v2.6.13


## 4. Create Collections

Collections are similar to tables in a relational database. Each collection has:
- **id**: Primary key field (INT64)
- **embedding**: Vector field for similarity search (FLOAT_VECTOR)
- **text**: Metadata field to store associated text (VARCHAR)

In [43]:
def create_collection(collection_name, drop_if_exists=True):
    """Create a collection with schema"""
    try:
        if utility.has_collection(collection_name):
            if drop_if_exists:
                utility.drop_collection(collection_name)
                print(f'Dropped existing collection: {collection_name}')
            else:
                print(f'Collection {collection_name} already exists')
                return False
        
        fields = [
            FieldSchema(name='id', dtype=DataType.INT64, is_primary=True),
            FieldSchema(name='embedding', dtype=DataType.FLOAT_VECTOR, dim=128),
            FieldSchema(name='text', dtype=DataType.VARCHAR, max_length=500)
        ]
        schema = CollectionSchema(fields=fields, description=f'Collection: {collection_name}')
        Collection(name=collection_name, schema=schema)
        print(f'Created collection: {collection_name}')
        return True
    except Exception as e:
        print(f'Failed to create collection: {e}')
        return False

for i in range(3, 6):
    collection_name = f'test_collection_{i}'
    create_collection(collection_name)

Dropped existing collection: test_collection_3
Created collection: test_collection_3
Dropped existing collection: test_collection_4
Created collection: test_collection_4
Dropped existing collection: test_collection_5
Created collection: test_collection_5


## 5. List Existing Collections

View all collections currently stored in the Milvus server.

In [8]:
def list_collections():
    """List all collections in Milvus"""
    try:
        collections = utility.list_collections()
        print('Collections in Milvus:')
        for col in collections:
            print(f'  - {col}')
        return collections
    except Exception as e:
        print(f'Failed to list collections: {e}')
        return []

list_collections()

Collections in Milvus:
  - test_collection_3
  - test_collection_0
  - test_collection_1
  - test_collection_2
  - test_collection_4
  - test_collection_5


['test_collection_3',
 'test_collection_0',
 'test_collection_1',
 'test_collection_2',
 'test_collection_4',
 'test_collection_5']

## 6. Insert Data into Collection

Insert sample vectors and metadata into a collection. This demonstrates batch insertion with random embeddings.

In [50]:
def insert_sample_data(collection_name, num_vectors=100):
    """Insert sample vectors and metadata into a collection"""
    try:
        collection = Collection(collection_name)
        ids = [i for i in range(num_vectors)]
        embeddings = [[random.random() for _ in range(128)] for _ in range(num_vectors)]
        texts = [f'Document {i}: Sample text content for vector {i}' for i in range(num_vectors)]
        
        data = [ids, embeddings, texts]
        collection.insert(data)
        print(f'Inserted {num_vectors} vectors into {collection_name}')
        return True
    except Exception as e:
        print(f'Failed to insert data: {e}')
        return False

insert_sample_data('test_collection_1', num_vectors=100)

Inserted 100 vectors into test_collection_1


True

## 7. Create Index for Vector Search

Indexes improve search performance. We create an IVF_FLAT index which partitions vectors into clusters for faster similarity search.

In [26]:
def create_index(collection_name, field_name='embedding'):
    """Create index on embedding field for faster search"""
    try:
        collection = Collection(collection_name)
        index_params = {
            'metric_type': 'L2',
            'index_type': 'IVF_FLAT',
            'params': {'nlist': 128}
        }
        collection.create_index(field_name=field_name, index_params=index_params)
        print(f'Created IVF_FLAT index on {field_name}')
        return True
    except Exception as e:
        print(f'Failed to create index: {e}')
        return False

create_index('test_collection_1')

Created IVF_FLAT index on embedding


True

## 8. Load Collection into Memory

Load the collection into memory before performing search operations. This is required for similarity search.

In [27]:
def load_collection(collection_name):
    """Load collection into memory"""
    try:
        collection = Collection(collection_name)
        collection.load()
        print(f'Loaded collection into memory: {collection_name}')
        return True
    except Exception as e:
        print(f'Failed to load collection: {e}')
        return False

load_collection('test_collection_1')

Loaded collection into memory: test_collection_1


True

## 9. Perform Vector Similarity Search

Search for similar vectors in the collection. This is the core operation of a vector database.

In [28]:
def vector_similarity_search(collection_name, query_vector, top_k=5):
    """Perform similarity search on vectors"""
    try:
        collection = Collection(collection_name)
        search_params = {'metric_type': 'L2', 'params': {'nprobe': 10}}
        results = collection.search(
            data=[query_vector],
            anns_field='embedding',
            param=search_params,
            limit=top_k,
            output_fields=['text', 'id']
        )
        
        print(f'Found {len(results[0])} similar vectors:')
        for i, result in enumerate(results[0], 1):
            print(f'  {i}. ID: {result.id}, Distance: {result.distance:.4f}')
        return results
    except Exception as e:
        print(f'Search failed: {e}')
        return None

query_vector = [random.random() for _ in range(128)]
vector_similarity_search('test_collection_1', query_vector, top_k=5)

Found 5 similar vectors:
  1. ID: 88, Distance: 16.9165
  2. ID: 81, Distance: 17.2784
  3. ID: 12, Distance: 17.7005
  4. ID: 25, Distance: 18.5105
  5. ID: 55, Distance: 18.9691


data: [[{'id': 88, 'distance': 16.91646957397461, 'entity': {'text': 'Document 88: Sample text content for vector 88', 'id': 88}}, {'id': 81, 'distance': 17.278371810913086, 'entity': {'text': 'Document 81: Sample text content for vector 81', 'id': 81}}, {'id': 12, 'distance': 17.700481414794922, 'entity': {'text': 'Document 12: Sample text content for vector 12', 'id': 12}}, {'id': 25, 'distance': 18.51052474975586, 'entity': {'text': 'Document 25: Sample text content for vector 25', 'id': 25}}, {'id': 55, 'distance': 18.96907615661621, 'entity': {'text': 'Document 55: Sample text content for vector 55', 'id': 55}}]]

## 10. Hybrid Search (Vector + Metadata Filter)

Combine vector similarity search with metadata filtering for more precise results.

In [29]:
def hybrid_search_with_filter(collection_name, query_vector, filter_expr=None, top_k=5):
    """Perform hybrid search with optional filters"""
    try:
        collection = Collection(collection_name)
        search_params = {'metric_type': 'L2', 'params': {'nprobe': 10}}
        results = collection.search(
            data=[query_vector],
            anns_field='embedding',
            param=search_params,
            limit=top_k,
            expr=filter_expr,
            output_fields=['text', 'id']
        )
        
        print(f'Hybrid search found {len(results[0])} results')
        for i, result in enumerate(results[0], 1):
            print(f'  {i}. ID: {result.id}, Distance: {result.distance:.4f}')
        return results
    except Exception as e:
        print(f'Hybrid search failed: {e}')
        return None

filter_expr = 'id < 50'
hybrid_search_with_filter('test_collection_1', query_vector, filter_expr=filter_expr, top_k=5)

Hybrid search found 5 results
  1. ID: 12, Distance: 17.7005
  2. ID: 25, Distance: 18.5105
  3. ID: 16, Distance: 19.2224
  4. ID: 35, Distance: 19.4014
  5. ID: 45, Distance: 19.6134


data: [[{'id': 12, 'distance': 17.700481414794922, 'entity': {'text': 'Document 12: Sample text content for vector 12', 'id': 12}}, {'id': 25, 'distance': 18.51052474975586, 'entity': {'text': 'Document 25: Sample text content for vector 25', 'id': 25}}, {'id': 16, 'distance': 19.22235679626465, 'entity': {'text': 'Document 16: Sample text content for vector 16', 'id': 16}}, {'id': 35, 'distance': 19.40144920349121, 'entity': {'text': 'Document 35: Sample text content for vector 35', 'id': 35}}, {'id': 45, 'distance': 19.613361358642578, 'entity': {'text': 'Document 45: Sample text content for vector 45', 'id': 45}}]]

## 11. Get Collection Statistics

Retrieve information about a collection such as the number of entities and indexed fields.

In [31]:
def get_collection_stats(collection_name):
    """Get collection statistics"""
    try:
        collection = Collection(collection_name)
        num_entities = collection.num_entities
        print(f'Collection: {collection_name}')
        print(f'  Number of entities: {num_entities}')
        return num_entities
    except Exception as e:
        print(f'Failed to get stats: {e}')
        return None

get_collection_stats('test_collection_1')

Collection: test_collection_1
  Number of entities: 0


0

## 12. Delete Operations

Delete entities from a collection using ID-based filtering.

In [34]:
def delete_entities(collection_name, delete_expr):
    """Delete entities from collection using expression"""
    try:
        collection = Collection(collection_name)
        result = collection.delete(delete_expr)
        print(f'Deleted entities matching: {delete_expr}')
        return result
    except Exception as e:
        print(f'Failed to delete entities: {e}')
        return None

delete_entities('test_collection_3', 'id > 90')
get_collection_stats('test_collection_3')

Deleted entities matching: id > 90
Collection: test_collection_3
  Number of entities: 200


200

## 13. Batch Operations on Multiple Collections

Demonstrate handling multiple collections with consistent operations.

In [35]:
def batch_operations_on_collections():
    """Perform batch operations on multiple collections"""
    collections = ['test_collection_3', 'test_collection_4', 'test_collection_5']
    
    for col_name in collections:
        print(f'\n--- Processing {col_name} ---')
        if utility.has_collection(col_name):
            insert_sample_data(col_name, num_vectors=50)
            create_index(col_name)
            load_collection(col_name)
            get_collection_stats(col_name)

batch_operations_on_collections()


--- Processing test_collection_3 ---
Inserted 50 vectors into test_collection_3
Created IVF_FLAT index on embedding
Loaded collection into memory: test_collection_3
Collection: test_collection_3
  Number of entities: 200

--- Processing test_collection_4 ---
Inserted 50 vectors into test_collection_4
Created IVF_FLAT index on embedding
Loaded collection into memory: test_collection_4
Collection: test_collection_4
  Number of entities: 0

--- Processing test_collection_5 ---
Inserted 50 vectors into test_collection_5
Created IVF_FLAT index on embedding
Loaded collection into memory: test_collection_5
Collection: test_collection_5
  Number of entities: 0


## 14. Release and Cleanup

Release collections from memory and disconnect from the server when done.

In [38]:
def release_and_cleanup():
    """Release collections from memory and disconnect"""
    try:
        collections = utility.list_collections()
        for col_name in collections:
            try:
                collection = Collection(col_name)
                collection.release()
                print(f'Released {col_name} from memory')
            except:
                pass
        
        connections.disconnect(alias='default')
        print('Disconnected from Milvus')
    except Exception as e:
        print(f'Error during cleanup: {e}')

release_and_cleanup()

Error during cleanup: <ConnectionNotExistException: (code=1, message=should create connection first.)>


## Summary

This notebook covered the essential Milvus operations:

1. **Connection Management**: Connecting and disconnecting from the server
2. **Collection Management**: Creating, listing, and deleting collections
3. **Data Operations**: Inserting and deleting data
4. **Indexing**: Creating indexes to improve search performance
5. **Search Operations**: Vector similarity search and hybrid search with filters
6. **Statistics**: Retrieving collection metadata and statistics
7. **Resource Management**: Loading/releasing collections and cleanup

### Key Concepts:
- **Collections**: Similar to tables, store vectors and metadata
- **Embeddings**: High-dimensional vectors used for similarity comparison
- **Indexes**: Accelerate search operations (IVF_FLAT, HNSW, etc.)
- **Distance Metrics**: L2 (Euclidean), IP (Inner Product), Cosine similarity
- **Load/Release**: Collections must be loaded into memory before search

For more information, visit the [Milvus documentation](https://milvus.io/docs)